# UBL GenericCode: Validate Google Sheets Revision → ODS → .gc Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/google-sheets-history-cQ6AV/notebooks/validate-revision-ods-to-gc.ipynb)

This notebook validates the full pipeline:
1. **Download** specific Google Sheets revisions as ODS files
2. **Convert** ODS → GenericCode (.gc) using Crane-ods2obdgc + Saxon
3. **Compare** results against known-good .gc files already in the repo

## Why Colab?
The Google Sheets API blocks requests from many cloud IP ranges.
Colab runs on Google's own infrastructure, so it has unrestricted access.

## Step 0: Setup & Authentication

In [ ]:
# Authenticate with Google (Colab built-in OAuth)
from google.colab import auth
auth.authenticate_user()
print('Authenticated successfully!')

In [ ]:
# Get access token for Drive API calls
import google.auth
from google.auth.transport.requests import Request as AuthRequest

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive.readonly']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

In [ ]:
# Clone the repository (for tools: Saxon, Crane XSLT, etc.)
import os
REPO_DIR = '/content/ubl-gc'

if not os.path.exists(REPO_DIR):
    !git clone --branch claude/google-sheets-history-cQ6AV \
        https://github.com/kduvekot/ubl-gc.git {REPO_DIR}
    print(f'Cloned to {REPO_DIR}')
else:
    !cd {REPO_DIR} && git pull origin claude/google-sheets-history-cQ6AV
    print(f'Updated {REPO_DIR}')

# Verify key tools exist
SAXON_JAR = f'{REPO_DIR}/history/tools/saxon9he/saxon9he.jar'
CRANE_XSL = f'{REPO_DIR}/history/tools/Crane-ods2obdgc/Crane-ods2obdgc.xsl'
assert os.path.exists(SAXON_JAR), f'Missing: {SAXON_JAR}'
assert os.path.exists(CRANE_XSL), f'Missing: {CRANE_XSL}'
print(f'Saxon: {SAXON_JAR}')
print(f'Crane XSLT: {CRANE_XSL}')

# Check Java
!java -version 2>&1 | head -1

## Step 1: Download Revision ODS Files

Uses the **Drive API v2** `revisions/{id}` → `exportLinks` endpoint,
which returns revision-specific content (unlike v3 `files.export`).

In [ ]:
import json, hashlib, time
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError

# --- Configuration ---
SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

# Revisions that map to our V1-V10 workflow runs
REVISIONS = {
    'ubl25_library': [
        {'id': '1843', 'maps_to': 'V1+V2', 'desc': 'Pre-CSD02 initial'},
        {'id': '1868', 'maps_to': 'V3+V4', 'desc': 'Customs rewritten'},
        {'id': '1999', 'maps_to': 'V5+V6', 'desc': 'CSD02 official'},
        {'id': '2005', 'maps_to': 'V7-V10', 'desc': 'Post-CSD02'},
    ],
    'ubl25_documents': [
        {'id': '1793', 'maps_to': 'V1+V2', 'desc': 'Pre-CSD02 baseline'},
        {'id': '1803', 'maps_to': 'V3', 'desc': 'Before Nov 20 edit'},
        {'id': '1983', 'maps_to': 'V4', 'desc': 'Nov 20 14:00 edit'},
        {'id': '2190', 'maps_to': 'V5-V7', 'desc': 'Last before csd02'},
        {'id': '2200', 'maps_to': 'V8', 'desc': 'Jan 21 edit'},
        {'id': '2204', 'maps_to': 'V9+V10', 'desc': 'Feb 4 edit'},
    ],
}

ODS_MIME = 'application/x-vnd.oasis.opendocument.spreadsheet'
ODS_MIME_ALT = 'application/vnd.oasis.opendocument.spreadsheet'
OUTPUT_DIR = Path('/content/revision-ods')
OUTPUT_DIR.mkdir(exist_ok=True)

# Expected SHA256 hashes from existing repo data
EXPECTED_HASHES = {
    'ubl25_library/rev-1843.ods':   '739eb59af31fed532cb5447084133584d810778d80b6324e8c1cf393ce175b9d',
    'ubl25_library/rev-1868.ods':   '25dee244334cba9a2efba57e0a9931e615b2bca65bebd8607abbe086492f73a8',
    'ubl25_library/rev-1999.ods':   '1abcef85fc371fda08c6def02d2891dfe9b71746cd64af72ae2659be47650779',
    'ubl25_library/rev-2005.ods':   '81ca61be7f2c0f287df675e0c726f3a8387bfc198e46b9e2c7d97039fca77579',
    'ubl25_documents/rev-1793.ods': '3d62ba6768897052e47b21b9692aee982c6f369d4a7a1f5eb992214ea20ea94d',
    'ubl25_documents/rev-1803.ods': 'aec99ad9aa64ddcdcec9f45978664fcc94a3b320672246cdccc6444e74300a39',
    'ubl25_documents/rev-1983.ods': '409deb8070a193ff525f28a3b6db2bc9bbcb0e4a97d02f4f38709034ea312ad0',
    'ubl25_documents/rev-2190.ods': 'f33ed101ada7656f113691d35f1291b33c59028d9b351d5f7f1fb0df1dbfe13f',
    'ubl25_documents/rev-2200.ods': 'a003bcafe5248fcf862738cbc9a505e6fc2049d0bbee5ac52bb54fbf0c3f13e2',
    'ubl25_documents/rev-2204.ods': '4213671be510eba75be14d16e4c29dea9db99f02bd5dfdfab64f4b500e23ac9c',
}

def sha256(data):
    return hashlib.sha256(data).hexdigest()

def api_get(url, binary=False):
    """Authenticated GET with retry."""
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(4):
        req = Request(url, headers=headers)
        try:
            with urlopen(req, timeout=120) as resp:
                data = resp.read()
                return resp.status, data if binary else json.loads(data)
        except HTTPError as e:
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  Rate limited ({e.code}), waiting {wait}s...')
                time.sleep(wait)
                continue
            return e.code, e.read().decode(errors='replace')
    return 0, 'max retries exceeded'

print(f'Configuration OK: {sum(len(v) for v in REVISIONS.values())} revisions to download')

In [ ]:
# Download each revision as ODS via Drive API v2 exportLinks
results = []

for sheet_key, file_id in SHEETS.items():
    sheet_dir = OUTPUT_DIR / sheet_key
    sheet_dir.mkdir(exist_ok=True)
    revisions = REVISIONS.get(sheet_key, [])

    print(f'\n=== {sheet_key} ({len(revisions)} revisions) ===')

    for rev in revisions:
        rev_id = rev['id']
        ods_path = sheet_dir / f'rev-{rev_id}.ods'
        rel_path = f'{sheet_key}/rev-{rev_id}.ods'

        print(f'  rev-{rev_id} [{rev["maps_to"]}]: {rev["desc"]}')

        # Skip if already downloaded
        if ods_path.exists() and ods_path.stat().st_size > 0:
            h = sha256(ods_path.read_bytes())
            expected = EXPECTED_HASHES.get(rel_path, '?')
            match = 'MATCH' if h == expected else 'MISMATCH'
            print(f'    [skip] exists, sha256={h[:16]}... {match}')
            results.append({'rev': rev_id, 'sheet': sheet_key, 'status': 'skipped',
                          'hash': h, 'match': match, 'size': ods_path.stat().st_size})
            continue

        # Step 1: Get exportLinks from Drive API v2
        v2_url = f'https://www.googleapis.com/drive/v2/files/{file_id}/revisions/{rev_id}'
        status, data = api_get(v2_url)
        if status != 200 or not isinstance(data, dict):
            print(f'    ERROR getting exportLinks: HTTP {status}')
            results.append({'rev': rev_id, 'sheet': sheet_key, 'status': 'failed'})
            time.sleep(1)
            continue

        export_links = data.get('exportLinks', {})
        ods_url = export_links.get(ODS_MIME) or export_links.get(ODS_MIME_ALT)
        if not ods_url:
            print(f'    ERROR: No ODS export link. Available: {list(export_links.keys())}')
            results.append({'rev': rev_id, 'sheet': sheet_key, 'status': 'no_ods_link'})
            time.sleep(1)
            continue

        print(f'    exportLinks OK ({len(export_links)} formats)')
        time.sleep(0.5)

        # Step 2: Download ODS
        status, ods_data = api_get(ods_url, binary=True)
        if status != 200 or not isinstance(ods_data, bytes):
            print(f'    ERROR downloading ODS: HTTP {status}')
            results.append({'rev': rev_id, 'sheet': sheet_key, 'status': 'download_failed'})
            time.sleep(1)
            continue

        # Save and verify
        ods_path.write_bytes(ods_data)
        h = sha256(ods_data)
        expected = EXPECTED_HASHES.get(rel_path, '?')
        match = 'MATCH' if h == expected else 'MISMATCH'
        print(f'    Downloaded: {len(ods_data):,} bytes, sha256={h[:16]}... {match}')

        results.append({'rev': rev_id, 'sheet': sheet_key, 'status': 'ok',
                      'hash': h, 'match': match, 'size': len(ods_data)})
        time.sleep(1)

print('\n=== Download Summary ===')
for r in results:
    status_icon = {'ok': '\u2705', 'skipped': '\u23ed\ufe0f', 'failed': '\u274c'}.get(r['status'], '\u2753')
    match_info = f" [{r.get('match', '?')}]" if 'match' in r else ''
    print(f"  {status_icon} {r['sheet']} rev-{r['rev']}: {r['status']}{match_info}")

## Step 2: Convert ODS → GenericCode (.gc)

Uses the exact same pipeline as `work-sheets/scripts/convert-revision-ods-to-gc.sh`:
1. Saxon + Crane-ods2obdgc.xsl + massageModelName.xml → UBL-Entities-2.5.gc
2. Saxon + Crane-ods2obdgc.xsl → raw endorsed .gc
3. Saxon + gc2endorsed.xsl → UBL-Endorsed-Entities-2.5.gc

In [ ]:
import subprocess, tempfile, shutil

SAXON_JAR = f'{REPO_DIR}/history/tools/saxon9he/saxon9he.jar'
CRANE_XSL = f'{REPO_DIR}/history/tools/Crane-ods2obdgc/Crane-ods2obdgc.xsl'
MASSAGE_XML = f'{REPO_DIR}/work-sheets/scripts/massageModelName.xml'
GC2ENDORSED_XSL = f'{REPO_DIR}/work-sheets/scripts/gc2endorsed.xsl'

# Same sheet-name filter as official build (excludes "Logs" sheets)
SHEET_REGEX = r'^([Ll]($|[^o].*|o($|[^g].*|g($|[^s].*))))|^[^Ll].*'

# V1-V10 mapping: (library_rev, documents_rev, stage_short, stage_dir)
VERSIONS = {
    'V1':  ('1843', '1793', 'CSD02', 'csd02'),
    'V2':  ('1843', '1793', 'CSD02', 'csd02'),
    'V3':  ('1868', '1803', 'CSD02', 'csd02'),
    'V4':  ('1868', '1983', 'CSD02', 'csd02'),
    'V5':  ('1999', '2190', 'CSD02', 'csd02'),
    'V6':  ('1999', '2190', 'CSD03', 'csd03'),
    'V7':  ('2005', '2190', 'CSD03', 'csd03'),
    'V8':  ('2005', '2200', 'CSD03', 'csd03'),
    'V9':  ('2005', '2204', 'CSD03', 'csd03'),
    'V10': ('2005', '2204', 'CSD03', 'csd03'),
}

# Expected .gc hashes for key versions
EXPECTED_GC = {
    'V1/UBL-Entities-2.5.gc':           'b62566116b2c302a37bdea8bc5b641b362cafbf8577fc656045248f1fcefca2b',
    'V1/UBL-Endorsed-Entities-2.5.gc':  '68a1bc6c9243aa787c8902dcb9f7ee6ed430aea8af1da869fd9aea7da335a3c1',
    'V5/UBL-Entities-2.5.gc':           'fa9822e180f7111bcd7286c4341cd70962befe12ccc36b518da26930b857a7bc',
    'V5/UBL-Endorsed-Entities-2.5.gc':  '0c9365e918f398aa8d564c94a8bb2280d934663cc5a15eb900c03b3ac357dcb0',
    'V10/UBL-Entities-2.5.gc':          'c3737612e83e59c482f6b513e11447f61df70e6f7fa72cdab09743daa6218070',
    'V10/UBL-Endorsed-Entities-2.5.gc': '3e6ba659d0e1970c020aeef122ab26b8d772db5f9fe42c866343d351421e4d9c',
}

GC_OUTPUT_DIR = Path('/content/gc-from-revisions')


def make_ident_xml(tmpdir, short, sdir, endorsed=False):
    """Generate identification XML matching official format."""
    suffix = '-Endorsed' if endorsed else ''
    name_suffix = ' Endorsed' if endorsed else ''
    uri_suffix = ':ENDORSED' if endorsed else ''
    file_suffix = '-Endorsed' if endorsed else ''

    xml = f"""<?xml version="1.0" encoding="UTF-8"?>
<Identification>
  <ShortName>UBL-2.5-{short}{suffix}</ShortName>
  <LongName>UBL 2.5 {short}{name_suffix} Business Entity Summary</LongName>
  <Version>2.5</Version>
  <CanonicalUri>urn:oasis:names:specification:ubl:BIE{uri_suffix}</CanonicalUri>
  <CanonicalVersionUri>urn:oasis:names:specification:ubl:BIE{uri_suffix}:2.5</CanonicalVersionUri>
  <LocationUri>http://docs.oasis-open.org/ubl/{sdir}-UBL-2.5/mod/UBL-Entities-2.5{file_suffix}.gc</LocationUri>
  <Agency>
     <LongName xml:lang="en">OASIS Universal Business Language</LongName>
     <Identifier>UBL</Identifier>
  </Agency>
</Identification>"""
    fname = 'ident-UBL-Endorsed.xml' if endorsed else 'ident-UBL.xml'
    path = os.path.join(tmpdir, fname)
    with open(path, 'w') as f:
        f.write(xml)
    return path


def run_saxon(args, label=''):
    """Run Saxon with given arguments, return (success, output)."""
    cmd = ['java', '-jar', SAXON_JAR] + args
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if result.returncode != 0:
        print(f'    SAXON FAILED ({label}): {result.stderr[-300:]}')
        return False
    return True


print(f'Will convert {len(VERSIONS)} versions')
print(f'Output: {GC_OUTPUT_DIR}/')

In [ ]:
# Convert each version
gc_results = []

for ver, (lib_rev, doc_rev, short, sdir) in VERSIONS.items():
    lib_ods = OUTPUT_DIR / 'ubl25_library' / f'rev-{lib_rev}.ods'
    doc_ods = OUTPUT_DIR / 'ubl25_documents' / f'rev-{doc_rev}.ods'
    out_dir = GC_OUTPUT_DIR / ver
    out_dir.mkdir(parents=True, exist_ok=True)

    entities_out = out_dir / 'UBL-Entities-2.5.gc'
    endorsed_out = out_dir / 'UBL-Endorsed-Entities-2.5.gc'

    print(f'\n--- {ver} (lib=rev-{lib_rev}, doc=rev-{doc_rev}, stage={short}) ---')

    if not lib_ods.exists():
        print(f'  ERROR: {lib_ods} not found')
        gc_results.append({'ver': ver, 'status': 'missing_ods'})
        continue
    if not doc_ods.exists():
        print(f'  ERROR: {doc_ods} not found')
        gc_results.append({'ver': ver, 'status': 'missing_ods'})
        continue

    # Create temp dir with ODS files and support files
    tmpdir = tempfile.mkdtemp()
    shutil.copy2(str(lib_ods), os.path.join(tmpdir, 'UBL-Library-Google.ods'))
    shutil.copy2(str(doc_ods), os.path.join(tmpdir, 'UBL-Documents-Google.ods'))
    shutil.copy2(MASSAGE_XML, os.path.join(tmpdir, 'massageModelName.xml'))

    ods_list = f"{tmpdir}/UBL-Library-Google.ods,{tmpdir}/UBL-Documents-Google.ods"

    # Generate ident files
    ident_main = make_ident_xml(tmpdir, short, sdir, endorsed=False)
    ident_endorsed = make_ident_xml(tmpdir, short, sdir, endorsed=True)

    # Step 1: Generate UBL-Entities-2.5.gc
    print(f'  [1/3] Generating UBL-Entities-2.5.gc...')
    ok = run_saxon([
        f'-xsl:{CRANE_XSL}',
        f'-o:{entities_out}',
        '-it:ods-uri',
        f'ods-uri={ods_list}',
        f'identification-uri={ident_main}',
        f'included-sheet-name-regex={SHEET_REGEX}',
        f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
    ], label='Entities')

    if not ok or not entities_out.exists():
        print(f'  FAILED: Entities conversion')
        gc_results.append({'ver': ver, 'status': 'failed_entities'})
        shutil.rmtree(tmpdir)
        continue

    e_size = entities_out.stat().st_size
    e_hash = sha256(entities_out.read_bytes())
    e_expected = EXPECTED_GC.get(f'{ver}/UBL-Entities-2.5.gc', '?')
    e_match = 'MATCH' if e_hash == e_expected else ('MISMATCH' if e_expected != '?' else 'no-ref')
    print(f'       {e_size:,} bytes, sha256={e_hash[:16]}... {e_match}')

    # Step 2: Generate raw endorsed .gc
    print(f'  [2/3] Generating endorsed raw...')
    raw_endorsed = os.path.join(tmpdir, 'raw-endorsed.gc')
    ok = run_saxon([
        f'-xsl:{CRANE_XSL}',
        f'-o:{raw_endorsed}',
        '-it:ods-uri',
        f'ods-uri={ods_list}',
        f'identification-uri={ident_endorsed}',
        f'included-sheet-name-regex={SHEET_REGEX}',
        f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
    ], label='Endorsed-raw')

    if not ok or not os.path.exists(raw_endorsed):
        print(f'  FAILED: Raw endorsed conversion')
        gc_results.append({'ver': ver, 'status': 'failed_endorsed_raw'})
        shutil.rmtree(tmpdir)
        continue

    # Step 3: Filter endorsed via gc2endorsed.xsl
    print(f'  [3/3] Filtering endorsed...')
    ok = run_saxon([
        f'-o:{endorsed_out}',
        f'-s:{raw_endorsed}',
        f'-xsl:{GC2ENDORSED_XSL}',
    ], label='Endorsed-filter')

    if not ok or not endorsed_out.exists():
        print(f'  FAILED: Endorsed filtering')
        gc_results.append({'ver': ver, 'status': 'failed_endorsed_filter'})
        shutil.rmtree(tmpdir)
        continue

    n_size = endorsed_out.stat().st_size
    n_hash = sha256(endorsed_out.read_bytes())
    n_expected = EXPECTED_GC.get(f'{ver}/UBL-Endorsed-Entities-2.5.gc', '?')
    n_match = 'MATCH' if n_hash == n_expected else ('MISMATCH' if n_expected != '?' else 'no-ref')
    print(f'       {n_size:,} bytes, sha256={n_hash[:16]}... {n_match}')

    gc_results.append({
        'ver': ver, 'status': 'ok',
        'entities_hash': e_hash, 'entities_match': e_match,
        'endorsed_hash': n_hash, 'endorsed_match': n_match,
        'entities_size': e_size, 'endorsed_size': n_size,
    })
    shutil.rmtree(tmpdir)

print('\n' + '='*60)
print('CONVERSION SUMMARY')
print('='*60)
for r in gc_results:
    if r['status'] == 'ok':
        icon = '\u2705' if r['entities_match'] == 'MATCH' and r['endorsed_match'] == 'MATCH' else '\u26a0\ufe0f'
        print(f"  {icon} {r['ver']}: Entities={r['entities_match']}, Endorsed={r['endorsed_match']}")
    else:
        print(f"  \u274c {r['ver']}: {r['status']}")

## Step 3: Results & Comparison

If the hashes match, we've proven that:
1. Colab can download revision-specific ODS from Google Sheets
2. The Crane pipeline produces identical .gc output
3. The full pipeline is reproducible and ready to scale to all 2,005 revisions

In [ ]:
# Final summary with sizes
import pandas as pd

rows = []
for r in gc_results:
    if r['status'] == 'ok':
        rows.append({
            'Version': r['ver'],
            'Entities Size': f"{r['entities_size']:,}",
            'Entities Match': r['entities_match'],
            'Endorsed Size': f"{r['endorsed_size']:,}",
            'Endorsed Match': r['endorsed_match'],
        })

if rows:
    df = pd.DataFrame(rows)
    display(df)
else:
    print('No successful conversions to display')

# Count matches
total = len([r for r in gc_results if r['status'] == 'ok'])
matches = len([r for r in gc_results if r['status'] == 'ok'
               and r.get('entities_match') == 'MATCH'
               and r.get('endorsed_match') == 'MATCH'])
no_ref = len([r for r in gc_results if r['status'] == 'ok'
              and 'no-ref' in (r.get('entities_match', ''), r.get('endorsed_match', ''))])
mismatches = total - matches - no_ref

print(f'\nResults: {matches} exact matches, {no_ref} no reference, {mismatches} mismatches out of {total}')

if mismatches == 0:
    print('\n*** Pipeline validated! Ready to scale to all revisions. ***')
else:
    print('\n*** Some mismatches detected — investigate before scaling. ***')

## Step 4 (Optional): Save Results to Google Drive

Save the ODS files and .gc outputs to Drive so they can be downloaded from anywhere.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create output folder on Drive
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy ODS files
ods_drive = DRIVE_DIR / 'revision-ods'
ods_drive.mkdir(exist_ok=True)
!cp -r /content/revision-ods/* {ods_drive}/

# Copy .gc files
gc_drive = DRIVE_DIR / 'gc-from-revisions'
gc_drive.mkdir(exist_ok=True)
!cp -r /content/gc-from-revisions/* {gc_drive}/

# Create compressed archive too
!cd /content && tar czf {DRIVE_DIR}/revision-validation.tar.gz revision-ods/ gc-from-revisions/

archive_size = (DRIVE_DIR / 'revision-validation.tar.gz').stat().st_size
print(f'\nSaved to Google Drive: {DRIVE_DIR}')
print(f'Archive: {archive_size:,} bytes ({archive_size/1024/1024:.1f} MB)')
print(f'\nYou can share the Drive folder to make it publicly accessible.')

---

## Next Steps

Once this validation succeeds, the next notebook will:
1. Download **all 2,005** Library revisions as ODS
2. Hash to find unique content states (~100-300 expected)
3. Convert unique states to .gc
4. Build a complete revision timeline

Estimated data:
- ODS: ~1.2 GB (2,005 x 625 KB)
- .gc (gzipped): ~735 MB (2,005 x 377 KB)
- .gc (unique only, gzipped): ~74 MB